In [ ]:
%pip install -q rapidfuzz jieba pandas

In [ ]:
# Metric
# 1. OCR confidence score
# 2. Agreement score

# 3. CometKiwi
# 4. LLM as a judge

In [ ]:

from pathlib import Path
import pandas as pd
import re
import unicodedata
import jieba

DATA_DIR = Path("/kaggle/input/datasets/bamboonguyen/tcm-evaluation")
PADDLE_PATH = DATA_DIR / "paddleocr_vl.txt"
HUYNQUAN_PATH = DATA_DIR / "huynquan_ocr.txt"
TRANSLATION_PATH = DATA_DIR / "translation vi.txt"

OUTPUT_DIR = Path("/kaggle/working/evaluation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def read_txt(path):
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")
    return path.read_text(encoding="utf-8-sig").strip()

paddle_text = read_txt(PADDLE_PATH)
huynquan_text = read_txt(HUYNQUAN_PATH)
translation_text = read_txt(TRANSLATION_PATH)

assert paddle_text, "PaddleOCR output is empty"
assert huynquan_text, "HuynquanOCR output is empty"
assert translation_text, "Translation is empty"

print("PaddleOCR characters:", len(paddle_text))
print("HuynquanOCR characters:", len(huynquan_text))
print("Vietnamese characters:", len(translation_text))


In [ ]:

def normalize_ocr(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", "", text)
    return text

paddle_norm = normalize_ocr(paddle_text)
huynquan_norm = normalize_ocr(huynquan_text)

print("PaddleOCR:", paddle_norm[:200])
print("HuynquanOCR:", huynquan_norm[:200])


In [ ]:

from rapidfuzz.distance import Levenshtein

def normalize_word_text(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text)).strip()

def tokenize_chinese(text):
    text = normalize_word_text(text)
    return [token.strip() for token in jieba.lcut(text)
            if token.strip()]

def agreement_metrics(text_a, text_b):
    a = normalize_ocr(text_a)
    b = normalize_ocr(text_b)

    char_ed = Levenshtein.distance(a, b)

    char_agreement = 1 - (
        char_ed / max(len(a), len(b), 1)
    )

    relative_cer = char_ed / max(len(a), 1)

    words_a = tokenize_chinese(text_a)
    words_b = tokenize_chinese(text_b)

    word_ed = Levenshtein.distance(words_a, words_b)

    word_agreement = 1 - (
        word_ed / max(len(words_a), len(words_b), 1)
    )

    relative_wer = word_ed / max(len(words_a), 1)

    return {
        "char_edit_distance": char_ed,
        "relative_cer": relative_cer,
        "char_agreement": char_agreement,
        "word_edit_distance": word_ed,
        "relative_wer": relative_wer,
        "word_agreement": word_agreement,
        "paddle_char_count": len(a),
        "huynquan_char_count": len(b),
        "paddle_word_count": len(words_a),
        "huynquan_word_count": len(words_b)
    }

ocr_results = agreement_metrics(paddle_text, huynquan_text)

display(pd.DataFrame([ocr_results]).T)


In [ ]:

from difflib import SequenceMatcher

matcher = SequenceMatcher(
    None,
    paddle_norm,
    huynquan_norm,
    autojunk=False
)

differences = []

for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == "equal":
        continue

    differences.append({
        "type": tag,
        "paddle_start": i1,
        "huynquan_start": j1,
        "paddle_text": paddle_norm[i1:i2],
        "huynquan_text": huynquan_norm[j1:j2]
    })

diff_df = pd.DataFrame(
    differences,
    columns=[
        "type", "paddle_start", "huynquan_start",
        "paddle_text", "huynquan_text"
    ]
)

display(diff_df.head(30))

diff_df.to_csv(
    OUTPUT_DIR / "ocr_differences.csv",
    index=False,
    encoding="utf-8-sig"
)


# CommetKiwi

In [ ]:
%pip install -q "sentence-transformers>=3,<6" pandas
%pip install --no-cache-dir --force-reinstall "protobuf==5.29.5"


In [ ]:

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

login(token=hf_token)
print("Hugging Face authentication completed")


In [ ]:
import sys
from importlib.metadata import version

print("Python:", sys.version)

for pkg in ["tensorflow", "protobuf", "keras"]:
    try:
        print(pkg, version(pkg))
    except Exception as e:
        print(pkg, e)

In [ ]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/multilingual-e5-small"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    MODEL_NAME,
    device=device
)

print("Device:", device)

In [ ]:
def split_segments(text):
    return [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

paddle_segments = split_segments(paddle_text)
huynquan_segments = split_segments(huynquan_text)
vi_segments = split_segments(translation_text)

print("Paddle segments:", len(paddle_segments))
print("Huynquan segments:", len(huynquan_segments))
print("Vietnamese segments:", len(vi_segments))

assert (
    len(paddle_segments)
    == len(huynquan_segments)
    == len(vi_segments)
), "Segments are not aligned. Manual alignment required."

df = pd.DataFrame({
    "paddle": paddle_segments,
    "huynquan": huynquan_segments,
    "vi": vi_segments
})

display(df.head())

In [ ]:
def get_embeddings(texts):
    texts = ["query: " + text for text in texts]

    return model.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    )

paddle_embeddings = get_embeddings(df["paddle"].tolist())
huynquan_embeddings = get_embeddings(df["huynquan"].tolist())
vi_embeddings = get_embeddings(df["vi"].tolist())

df["similarity_paddle_vi"] = np.sum(
    paddle_embeddings * vi_embeddings,
    axis=1
)

df["similarity_huynquan_vi"] = np.sum(
    huynquan_embeddings * vi_embeddings,
    axis=1
)

df["similarity_gap"] = np.abs(
    df["similarity_paddle_vi"]
    - df["similarity_huynquan_vi"]
)

display(df.head())

In [ ]:
metrics = [
    "similarity_paddle_vi",
    "similarity_huynquan_vi",
    "similarity_gap"
]

summary = df[metrics].agg([
    "mean",
    "std",
    "median",
    "min",
    "max"
])

display(summary)

df.to_csv(
    OUTPUT_DIR / "translation_similarity.csv",
    index=False,
    encoding="utf-8-sig"
)